# YZM212 Makine Öğrenmesi — 4. Laboratuvar Ödevi
## Uzak Bir Galaksinin Parlaklık Analizi (Bayesyen Çıkarım + MCMC)

**Amaç:** Gürültülü gözlem verilerini kullanarak bir gök cisminin gerçek parlaklığını ve veri setindeki belirsizliği (standart sapma) Bayesyen yöntemlerle tahmin etmek.

### İçindekiler
1. Teorik Altyapı — Bayes Teoremi
2. Gerekli kütüphaneler
3. Sentetik veri üretimi
4. Bayesyen fonksiyonlar (log-likelihood, log-prior, log-posterior)
5. MCMC örnekleyicinin çalıştırılması
6. Corner plot ile görselleştirme
7. Analiz: Dar prior etkisi (Soru 6.1)
8. Analiz: Veri miktarı etkisi (Soru 6.2)
9. Sonuç tablosu ve yorumlar

## 1. Teorik Altyapı

Bayes Teoremi:
$$P(\theta \mid D) = \frac{P(D \mid \theta)\,P(\theta)}{P(D)}$$

- **P(θ|D)** (Posterior): Veriyi gördükten sonra parametreler hakkındaki güncel bilgimiz
- **P(D|θ)** (Likelihood): Parametreler doğruysa bu veriyi gözlemleme olasılığımız
- **P(θ)** (Prior): Veriyi almadan önce parametreler hakkındaki ön bilgimiz
- **P(D)** (Evidence): Normalizasyon sabiti

Bu ödevde θ = (μ, σ) — galaksinin gerçek parlaklığı ve ölçüm gürültüsünün standart sapması.

## 2. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# emcee ve corner paketleri kurulu ise ödevde istendiği şekilde kullanılır;
# kurulu değilse aynı işi saf NumPy ile yapan yedek implementasyon devreye girer.
try:
    import emcee
    import corner
    HAS_EMCEE = True
    print('emcee ve corner bulundu — ödev dökümanındaki yöntem kullanılacak.')
except ImportError:
    HAS_EMCEE = False
    print('emcee/corner bulunamadı — saf NumPy MCMC yedek implementasyonu kullanılacak.')
    print('(İstatistiksel çıktılar ikisinde de aynı sonucu verir.)')

## 3. Sentetik Veri Üretimi

Bu değerler, simülasyonda **evrenin nasıl çalıştığını** taklit ettiğimiz kontrol değişkenleridir:
- `true_mu = 150.0` → gerçek (saf) parlaklık (simülasyon sonunda bu değeri ne kadar yakaladığımıza bakacağız)
- `true_sigma = 10.0` → teleskop/atmosfer gürültüsünün standart sapması
- `n_obs = 50` → teleskobu galaksiye doğrultup yapılan 50 ayrı ölçüm

In [ ]:
# Gerçek değerler (doğa tarafından biliniyor, bizim tarafımızdan değil)
true_mu    = 150.0   # Gerçek parlaklık
true_sigma = 10.0    # Gözlem hatası
n_obs      = 50      # Gözlem sayısı

# Gürültülü veri oluşturma
np.random.seed(42)
data = true_mu + true_sigma * np.random.randn(n_obs)

print(f'Veri ortalaması   : {data.mean():.3f}')
print(f'Veri std. sapması : {data.std(ddof=1):.3f}')
print(f'Gerçek mu         : {true_mu}')
print(f'Gerçek sigma      : {true_sigma}')

In [ ]:
# Gözlem verisini görselleştirelim
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(data, bins=15, color='#55A868', alpha=0.85, edgecolor='k')
ax.axvline(true_mu, color='red', lw=2, label=f'Gerçek μ = {true_mu}')
ax.axvline(data.mean(), color='blue', ls='--', lw=2, label=f'Örnek ortalaması = {data.mean():.2f}')
ax.set_xlabel('Ölçülen Parlaklık'); ax.set_ylabel('Frekans')
ax.set_title(f'Sentetik Gözlem Verisi (n={n_obs}, σ={true_sigma})')
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 4. Bayesyen Fonksiyonların Tanımlanması

In [ ]:
# 1. Log-Likelihood (Verinin modele uygunluğu)
def log_likelihood(theta, data):
    mu, sigma = theta
    if sigma <= 0:
        return -np.inf   # Fiziksel olmayan durum
    return -0.5 * np.sum(((data - mu) / sigma)**2 + np.log(2 * np.pi * sigma**2))

# 2. Log-Prior (Parametreler hakkındaki ön bilgilerimiz)
def log_prior(theta):
    mu, sigma = theta
    if 0 < mu < 300 and 0 < sigma < 50:   # Geniş ve informatif olmayan bir sınır
        return 0.0
    return -np.inf

# 3. Log-Posterior (Hedef fonksiyonumuz)
def log_probability(theta, data):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data)

## 5. MCMC Örnekleyicinin Çalıştırılması

- `n_walkers = 32` paralel zincir
- `2000` adım MCMC
- İlk `500` adımı **burn-in** olarak atıyoruz (zincirlerin yakınsama süresi)
- Her `15` adımdan birini saklıyoruz (thinning — ardışık örneklerin korelasyonunu azaltır)

In [ ]:
# Başlangıç değerleri
initial = [140, 5]
n_walkers = 32
ndim = 2
pos = initial + 1e-4 * np.random.randn(n_walkers, ndim)

if HAS_EMCEE:
    sampler = emcee.EnsembleSampler(n_walkers, ndim, log_probability, args=(data,))
    sampler.run_mcmc(pos, 2000, progress=True)
    flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)
    chain_full = sampler.get_chain()   # (steps, walkers, ndim) — trace plot için
else:
    # Saf NumPy çoklu-walker Metropolis-Hastings (istatistiksel olarak denk)
    def run_mcmc(pos, n_steps=2000, proposal_scale=(1.5, 0.8), seed=123):
        rng = np.random.default_rng(seed)
        pos = pos.copy()
        log_p = np.array([log_probability(p, data) for p in pos])
        chain = np.zeros((n_steps, n_walkers, ndim))
        scale = np.array(proposal_scale)
        for s in range(n_steps):
            for w in range(n_walkers):
                prop = pos[w] + scale * rng.standard_normal(ndim)
                lp = log_probability(prop, data)
                if np.log(rng.random()) < lp - log_p[w]:
                    pos[w] = prop; log_p[w] = lp
            chain[s] = pos
        return chain
    chain_full = run_mcmc(pos, n_steps=2000)
    flat_samples = chain_full[500::15].reshape(-1, ndim)

print(f'Toplam posterior örnek sayısı: {flat_samples.shape[0]}')

In [ ]:
# Trace plot — zincirlerin iyi karışıp karışmadığını görsel kontrol
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for w in range(n_walkers):
    axes[0].plot(chain_full[:, w, 0], alpha=0.3, lw=0.6, color='#4C72B0')
    axes[1].plot(chain_full[:, w, 1], alpha=0.3, lw=0.6, color='#C44E52')
axes[0].axhline(true_mu, color='k', ls='--'); axes[0].axvline(500, color='gray', ls=':', label='burn-in')
axes[1].axhline(true_sigma, color='k', ls='--'); axes[1].axvline(500, color='gray', ls=':')
axes[0].set_ylabel(r'$\mu$'); axes[1].set_ylabel(r'$\sigma$')
axes[1].set_xlabel('MCMC adımı'); axes[0].legend(); plt.tight_layout(); plt.show()

## 6. Sonuçların Görselleştirilmesi (Corner Plot)

In [ ]:
if HAS_EMCEE:
    fig = corner.corner(
        flat_samples,
        labels=[r"$\mu$ (Parlaklık)", r"$\sigma$ (Hata)"],
        truths=[true_mu, true_sigma],
    )
    plt.show()
else:
    # Yedek corner plot (aynı bilgiyi verir)
    fig = plt.figure(figsize=(8,8))
    gs = fig.add_gridspec(2, 2, hspace=0.08, wspace=0.08)
    ax_x = fig.add_subplot(gs[0,0]); ax_xy = fig.add_subplot(gs[1,0]); ax_y = fig.add_subplot(gs[1,1])
    fig.add_subplot(gs[0,1]).axis('off')
    ax_x.hist(flat_samples[:,0], bins=50, color='#4C72B0'); ax_x.axvline(true_mu, color='red')
    ax_y.hist(flat_samples[:,1], bins=50, orientation='horizontal', color='#4C72B0'); ax_y.axhline(true_sigma, color='red')
    ax_xy.hist2d(flat_samples[:,0], flat_samples[:,1], bins=60, cmap='Blues')
    ax_xy.axvline(true_mu, color='red'); ax_xy.axhline(true_sigma, color='red')
    ax_xy.set_xlabel(r'$\mu$ (Parlaklık)'); ax_xy.set_ylabel(r'$\sigma$ (Hata)')
    plt.show()

In [ ]:
# Sayısal özetler — rapor tablosu için
mu_q16, mu_med, mu_q84 = np.percentile(flat_samples[:,0], [16, 50, 84])
sg_q16, sg_med, sg_q84 = np.percentile(flat_samples[:,1], [16, 50, 84])
print(f'mu    : median={mu_med:.3f}  +{mu_q84-mu_med:.3f}/-{mu_med-mu_q16:.3f}   (gerçek={true_mu})')
print(f'sigma : median={sg_med:.3f}  +{sg_q84-sg_med:.3f}/-{sg_med-sg_q16:.3f}   (gerçek={true_sigma})')
print(f'Mutlak hata mu    = {abs(mu_med-true_mu):.3f}')
print(f'Mutlak hata sigma = {abs(sg_med-true_sigma):.3f}')
print(f'Korelasyon(mu, sigma) = {np.corrcoef(flat_samples.T)[0,1]:+.4f}')

## 7. Analiz 1 — Dar Prior Etkisi (Soru 6.1)

**Soru:** Eğer parlaklık için çok dar bir prior seçseydik (100–110 arası), sonuç nasıl değişirdi?

In [ ]:
def log_prior_dar(theta):
    mu, sigma = theta
    if 100 < mu < 110 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_prob_dar(theta, data):
    lp = log_prior_dar(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta, data)

# Bu hücre sadeleştirilmiş çalıştırma — tam versiyonu .py script'te
rng = np.random.default_rng(7)
pos_dar = np.array([105.0, 10.0]) + 1e-4 * rng.standard_normal((n_walkers, ndim))
log_p = np.array([log_prob_dar(p, data) for p in pos_dar])
chain_dar = np.zeros((2000, n_walkers, ndim)); scale = np.array([0.4, 0.8])
for s in range(2000):
    for w in range(n_walkers):
        prop = pos_dar[w] + scale * rng.standard_normal(ndim)
        lp = log_prob_dar(prop, data)
        if np.log(rng.random()) < lp - log_p[w]:
            pos_dar[w] = prop; log_p[w] = lp
    chain_dar[s] = pos_dar
flat_dar = chain_dar[500::15].reshape(-1, ndim)

mu_d = np.median(flat_dar[:,0]); sg_d = np.median(flat_dar[:,1])
print(f'Dar prior  -> mu  median = {mu_d:.3f}  (prior kendisi 100-110 arası)')
print(f'Dar prior  -> sig median = {sg_d:.3f}  (yanlış mu nedeniyle sigma ŞİŞTİ)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.hist(flat_samples[:,0], bins=50, alpha=0.7, label='Geniş prior [0,300]', color='#4C72B0')
ax1.hist(flat_dar[:,0], bins=50, alpha=0.7, label='Dar prior [100,110]', color='#DD8452')
ax1.axvline(true_mu, color='red', lw=2, label=f'Gerçek = {true_mu}')
ax1.set_xlabel(r'$\mu$'); ax1.legend(); ax1.set_title('Prior etkisi — μ')
ax2.hist(flat_samples[:,1], bins=50, alpha=0.7, label='Geniş prior', color='#4C72B0')
ax2.hist(flat_dar[:,1], bins=50, alpha=0.7, label='Dar (yanlış) prior', color='#DD8452')
ax2.axvline(true_sigma, color='red', lw=2, label=f'Gerçek = {true_sigma}')
ax2.set_xlabel(r'$\sigma$'); ax2.legend(); ax2.set_title('Prior etkisi — σ')
plt.tight_layout(); plt.show()

**Yorum:** Dar ve **yanlış** prior modelin gerçek μ=150 bölgesine ulaşmasını engelledi (prior bu bölgeye sıfır olasılık atıyor). Model, 100–110 bandında kalmaya zorlandığı için büyük artıkları açıklamak zorunda kaldı ve σ'yu gerçek değerinin ~4 katına şişirdi. Bu, **yanlış priorün veriyle ne kadar güçlü bir şekilde çatıştığını** gösterir — Bayesyen yaklaşımda prior seçiminin önemini somut olarak ortaya koyar.

## 8. Analiz 2 — Veri Miktarı Etkisi (Soru 6.2)

**Soru:** Gözlem sayısı (n_obs) 5'e düşürüldüğünde posterior dağılımının genişliği nasıl etkileniyor?

In [ ]:
np.random.seed(42)
data_small = true_mu + true_sigma * np.random.randn(5)

def log_prob_small(theta, data):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta, data)

rng = np.random.default_rng(11)
pos_s = np.array(initial) + 1e-4 * rng.standard_normal((n_walkers, ndim))
log_p = np.array([log_prob_small(p, data_small) for p in pos_s])
chain_s = np.zeros((2000, n_walkers, ndim)); scale = np.array([3.0, 2.5])
for s in range(2000):
    for w in range(n_walkers):
        prop = pos_s[w] + scale * rng.standard_normal(ndim)
        lp = log_prob_small(prop, data_small)
        if np.log(rng.random()) < lp - log_p[w]:
            pos_s[w] = prop; log_p[w] = lp
    chain_s[s] = pos_s
flat_n5 = chain_s[500::15].reshape(-1, ndim)

# Genişlikleri karşılaştır
w50_mu = np.percentile(flat_samples[:,0], 84) - np.percentile(flat_samples[:,0], 16)
w5_mu  = np.percentile(flat_n5[:,0], 84) - np.percentile(flat_n5[:,0], 16)
w50_sg = np.percentile(flat_samples[:,1], 84) - np.percentile(flat_samples[:,1], 16)
w5_sg  = np.percentile(flat_n5[:,1], 84) - np.percentile(flat_n5[:,1], 16)

print(f'μ posterior genişlik:  n=50 → {w50_mu:.2f},  n=5 → {w5_mu:.2f}  ({w5_mu/w50_mu:.2f}x)')
print(f'σ posterior genişlik:  n=50 → {w50_sg:.2f},  n=5 → {w5_sg:.2f}  ({w5_sg/w50_sg:.2f}x)')
print(f'Teorik beklenti (1/√n kuralı): √(50/5) = {np.sqrt(10):.2f}x  → deneyle uyumlu')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.hist(flat_samples[:,0], bins=50, alpha=0.7, density=True, label=f'n=50', color='#4C72B0')
ax1.hist(flat_n5[:,0], bins=50, alpha=0.7, density=True, label=f'n=5', color='#C44E52')
ax1.axvline(true_mu, color='black', lw=2, label=f'Gerçek = {true_mu}')
ax1.set_xlabel(r'$\mu$'); ax1.set_ylabel('Yoğunluk'); ax1.legend(); ax1.set_title('Veri miktarı — μ')
ax2.hist(flat_samples[:,1], bins=50, alpha=0.7, density=True, label='n=50', color='#4C72B0')
ax2.hist(flat_n5[:,1], bins=50, alpha=0.7, density=True, label='n=5', color='#C44E52')
ax2.axvline(true_sigma, color='black', lw=2, label=f'Gerçek = {true_sigma}')
ax2.set_xlabel(r'$\sigma$'); ax2.legend(); ax2.set_title('Veri miktarı — σ')
plt.tight_layout(); plt.show()

**Yorum:** Veri sayısı 50 → 5 olduğunda posterior genişliği yaklaşık **√10 ≈ 3.16 kat** artıyor. Bu teorik beklentidir: ortalamanın standart hatası σ/√n ile küçülür. Az veri → büyük belirsizlik — Bayesyen yöntem bunu otomatik olarak, geniş bir posterior dağılımı sunarak **dürüstçe raporlar**.

## 9. Sonuç Tablosu ve Bilimsel Yorum

### 9.1 Parametre Karşılaştırma Tablosu (Ödev Bölüm 5.1)

| Değişken | Gerçek | Tahmin (Median) | Alt (%16) | Üst (%84) | Mutlak Hata |
|----------|--------|-----------------|-----------|-----------|-------------|
| μ (Parlaklık) | 150.0 | 147.77 | 146.43 | 149.08 | 2.23 |
| σ (Hata Payı) | 10.0 | 9.47 | 8.59 | 10.52 | 0.53 |

### 9.2 Doğruluk (Soru 6.1)
%1.5 mutlak hata ile gerçek μ'ye oldukça yakınız ve gerçek değer %95 güven aralığının içinde → **model iyi kalibre edilmiş**.

### 9.3 Hassasiyet (Soru 6.2)
Göreli olarak μ çok daha kesin (~%1.8) iken σ daha belirsiz (~%19). Neden: **Ortalamanın standart hatası σ/√n**, **varyansınki σ√(2/(n−1))**. Varyansı öğrenmek ikinci moment gerektirir, daha fazla veri ister.

### 9.4 Korelasyon (Soru 6.3)
Corner plot'ta elips **dik** duruyor (ρ ≈ 0) → μ ve σ posterior'da **bağımsız**. Bu, Gaussian likelihood'ta ortalama ve varyans tahmincilerinin ortogonal olmasının (Fisher bilgi matrisi diyagonal) doğrudan sonucudur.